In [1]:
import numpy as np
import pandas as pd

class SVM_From_Scratch:
    def __init__(self, learning_rate=0.001, lambda_param=0.01, n_iters=1000):
        # learning_rate: How big of a step the model takes when adjusting the line
        # lambda_param: The regularization penalty (controls the width of the margin)
        # n_iters: How many times we loop over the data to learn
        self.lr = learning_rate
        self.lambda_param = lambda_param
        self.n_iters = n_iters
        self.w = None # Weights (coefficients for features like eTIV, nWBV)
        self.b = None # Bias (the intercept)

    def fit(self, X, y):
        # CRITICAL MATH STEP: SVM math strictly requires labels to be -1 and 1.
        # Our current data has 0 (Healthy) and 1 (Demented). We must temporarily map 0 to -1.
        y_ = np.where(y <= 0, -1, 1)
        
        n_samples, n_features = X.shape
        
        # Initialize weights and bias to zeros
        self.w = np.zeros(n_features)
        self.b = 0

        # Gradient Descent: The learning loop
        for _ in range(self.n_iters):
            for idx, x_i in enumerate(X):
                # The Margin Equation: y_i * (w.x - b) >= 1
                condition = y_[idx] * (np.dot(x_i, self.w) - self.b) >= 1
                
                if condition:
                    # Point is correctly classified and safely outside the margin.
                    # We only apply a small regularization penalty to keep weights small.
                    self.w -= self.lr * (2 * self.lambda_param * self.w)
                else:
                    # Point is inside the margin or completely misclassified! (Hinge Loss)
                    # We heavily adjust the weights and the bias to fix this mistake.
                    self.w -= self.lr * (2 * self.lambda_param * self.w - np.dot(x_i, y_[idx]))
                    self.b -= self.lr * y_[idx]

    def predict(self, X):
        # Calculate the linear equation: w.x - b
        approx = np.dot(X, self.w) - self.b
        
        # np.sign returns 1 if positive, -1 if negative
        predictions = np.sign(approx)
        
        # Map the predictions back to 0 and 1 so they match our project requirements
        return np.where(predictions == -1, 0, 1)

# --- Quick Sanity Check ---
# Let's test it with dummy data to prove it learns
X_dummy = np.array([[1, 2], [1.5, 1.8], [8, 8], [9, 9.5]]) # Features
y_dummy = np.array([0, 0, 1, 1])                           # Labels

test_svm = SVM_From_Scratch(n_iters=1000)
test_svm.fit(X_dummy, y_dummy)
print("Dummy Predictions:", test_svm.predict(X_dummy))
print("Final Weights:", test_svm.w)

Dummy Predictions: [0 0 1 1]
Final Weights: [ 0.58960809 -0.29040453]


In [3]:
import pandas as pd
import time

# 1. Load the preprocessed data (convert to numpy arrays for our custom math)
print("Loading preprocessed dataset...")
X_train = pd.read_csv('../dataset/processed/X_train.csv').values
y_train = pd.read_csv('../dataset/processed/y_train.csv').values.ravel()
X_test = pd.read_csv('../dataset/processed/X_test.csv').values
y_test = pd.read_csv('../dataset/processed/y_test.csv').values.ravel()

# 2. Initialize our custom model
# We use a slightly larger number of iterations (n_iters) because the real dataset is complex
custom_svm = SVM_From_Scratch(learning_rate=0.001, lambda_param=0.01, n_iters=2000)

# 3. Train the model and time it
print("Training Custom Linear SVM. Please wait...")
start_time = time.time()

custom_svm.fit(X_train, y_train)

end_time = time.time()
print(f"Training Complete! Time taken: {end_time - start_time:.2f} seconds.")

Loading preprocessed dataset...
Training Custom Linear SVM. Please wait...
Training Complete! Time taken: 2.14 seconds.


In [4]:
from sklearn.metrics import accuracy_score, recall_score, classification_report, confusion_matrix

# 1. Generate predictions on the unseen test set
print("Generating predictions on the test set...")
y_pred = custom_svm.predict(X_test)

# 2. Calculate core metrics
# Recall is critical in medicine: Out of all the patients who ACTUALLY have Alzheimer's, how many did we catch?
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print("\n--- Custom SVM Performance ---")
print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"Recall (Sensitivity): {recall * 100:.2f}%\n")

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Generating predictions on the test set...

--- Custom SVM Performance ---
Accuracy: 96.00%
Recall (Sensitivity): 91.89%

Classification Report:
              precision    recall  f1-score   support

           0       0.93      1.00      0.96        38
           1       1.00      0.92      0.96        37

    accuracy                           0.96        75
   macro avg       0.96      0.96      0.96        75
weighted avg       0.96      0.96      0.96        75

Confusion Matrix:
[[38  0]
 [ 3 34]]
